### __Import__

In [1]:
from data_model_loader import *

model = load_model()
images = load_coco_2014_dataset()

# load file paths
with open('config.json', 'r') as file:
    config = json.load(file)
coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']

filename_to_image_id = get_image_ids()


OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Randomly selecting 1500 pictures ..
Done!


### __Predict__

In [2]:
import base_model

results = base_model.predict(model, coco_folder, images, n_images=len(images))
results

Inferencing images: 100%|██████████| 1500/1500 [00:48<00:00, 30.83it/s]


[{'image_id': 144334,
  'category_id': 59,
  'bbox': [97.68460750579834,
   118.09185028076172,
   328.5994005203247,
   379.40589904785156],
  'score': 0.6847849488258362},
 {'image_id': 144334,
  'category_id': 67,
  'bbox': [2.3686838150024414,
   26.604480743408203,
   475.5811643600464,
   599.5688438415527],
  'score': 0.6462284326553345},
 {'image_id': 144334,
  'category_id': 59,
  'bbox': [185.07574081420898,
   110.12358665466309,
   199.59383010864258,
   176.10573768615723],
  'score': 0.6213268637657166},
 {'image_id': 144334,
  'category_id': 59,
  'bbox': [155.16995429992676,
   241.0403823852539,
   279.6841621398926,
   235.71182250976562],
  'score': 0.5659355521202087},
 {'image_id': 144334,
  'category_id': 48,
  'bbox': [118.1427526473999,
   18.418235778808594,
   47.05945014953613,
   53.504905700683594],
  'score': 0.3902663290500641},
 {'image_id': 144334,
  'category_id': 56,
  'bbox': [147.9569435119629,
   158.28486442565918,
   45.566396713256836,
   64.470

In [3]:
from PIL import Image
import torchvision.transforms as transforms
import pathlib
from tqdm import tqdm


def preprocess_image(image):
    # convert greyscale to rgb
    if image.mode == 'L':
        image = image.convert('RGB')

    transform = transforms.Compose([
        transforms.ToTensor()  # Converts the image to a tensor [C, H, W] with values between 0 and 1
    ])
    
    image_tensor = transform(image)
    image_tensor = (image_tensor * 255).byte()
    image_tensor = image_tensor.permute(1, 2, 0)  # [C, H, W] -> [H, W, C]   
    # Add a batch dimension [1, H, W, C]
    image_tensor = image_tensor.unsqueeze(0)
    
    return image_tensor

def predict_saved_model(model, data_path, images, n_images=5):
    data_path = pathlib.Path(data_path)
    results = []

    for img in tqdm(images[:n_images], desc="Loading images"):
        # load image
        image_path = data_path/"val2014"/"val2014"/img
        image = Image.open(image_path)
        width, height = image.size

        # transform
        image_tensor = preprocess_image(image)

        # inference
        detector_output = model(image_tensor)
              
        # evaluate
        n_detections = int(detector_output["num_detections"].numpy()[0])
        #print(detector_output["detection_boxes"])
        for i in range(n_detections):
            ymin, xmin, ymax, xmax = detector_output["detection_boxes"].numpy()[0][i]
            ymin = ymin * height
            ymax = ymax * height
            xmin = xmin * width
            xmax = xmax * width

            result = {
                    "image_id" : int(filename_to_image_id[img]),
                    "category_id":int(detector_output["detection_classes"].numpy()[0][i]),
                    "bbox": [xmin, ymin, xmax - xmin, ymax - ymin], # detector_output["detection_boxes"].numpy()[0][i].tolist(), # needs to be list
                    "score": float(detector_output["detection_scores"].numpy()[0][i])
            }
            results.append(result)
    
    return results
    


In [4]:
# load image
image_path = r"data\coco2014\val2014\val2014\COCO_val2014_000000322029.jpg"
image = Image.open(image_path)
width, height = image.size

image_tensor = preprocess_image(image)

# inference
detector_output = model(image_tensor)

detector_output

{'num_detections': <tf.Tensor: shape=(1,), dtype=float32, numpy=array([100.], dtype=float32)>,
 'detection_classes': <tf.Tensor: shape=(1, 100), dtype=float32, numpy=
 array([[ 1., 19.,  1.,  2.,  1.,  1.,  3.,  1.,  1.,  1.,  1.,  1.,  1.,
          1.,  1.,  1.,  3.,  1.,  1.,  3.,  1.,  1.,  8.,  3., 19.,  1.,
          1., 77.,  1.,  3., 77.,  1., 19.,  8.,  2.,  1.,  2.,  8.,  3.,
          1.,  1.,  4.,  1.,  2.,  1., 31., 77.,  1.,  2., 31., 31., 62.,
          3., 27.,  1.,  3.,  1.,  4.,  1.,  1.,  2.,  8.,  3., 77., 44.,
         31.,  3., 31., 84.,  3., 62.,  2.,  1., 31., 27.,  1.,  1.,  3.,
         77.,  1.,  1.,  1.,  1.,  1., 77.,  2.,  1.,  1., 62., 31., 31.,
         19., 33., 75., 44.,  1., 44., 75., 32., 19.]], dtype=float32)>,
 'raw_detection_boxes': <tf.Tensor: shape=(1, 1917, 4), dtype=float32, numpy=
 array([[[ 0.00103576,  0.00543218,  0.03236269,  0.06680968],
         [-0.01356092, -0.06478541,  0.06727028,  0.16343951],
         [-0.05822851, -0.01802536,  0

In [5]:
detector_output.keys()

dict_keys(['num_detections', 'detection_classes', 'raw_detection_boxes', 'detection_anchor_indices', 'raw_detection_scores', 'detection_boxes', 'detection_scores', 'detection_multiclass_scores'])

In [6]:
results = predict(model, coco_folder, images, n_images=2)
results

NameError: name 'predict' is not defined

In [ ]:
# create result dir
results_dir = pathlib.Path('results/')
results_dir.mkdir(exist_ok=True, parents=True)
results_file = results_dir/"results.json"

# store results
with open(results_file, 'w') as json_file:
    json.dump(results, json_file, indent=4)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
import skimage.io as io
import pylab
pylab.rcParams['figure.figsize'] = (10.0, 8.0)

annType = 'bbox'

#initialize COCO ground truth api
cocoGt=COCO(annotation_file_path)

#initialize COCO detections api
cocoDt=cocoGt.loadRes('results/results.json')



loading annotations into memory...
Done (t=3.07s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.40s)
creating index...
index created!


In [ ]:
imgIds=sorted(cocoGt.getImgIds())

imgIds=imgIds[0:100]
imgId = imgIds[np.random.randint(100)]

In [ ]:
cocoEval = COCOeval(cocoGt, cocoDt, annType)
cocoEval.params.imgIds = imgIds
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.13s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100

### Old

In [ ]:
classes = get_annotation(annotation_file_path)

In [ ]:
classes

{1: 'person',
 2: 'bicycle',
 3: 'car',
 4: 'motorcycle',
 5: 'airplane',
 6: 'bus',
 7: 'train',
 8: 'truck',
 9: 'boat',
 10: 'traffic light',
 11: 'fire hydrant',
 13: 'stop sign',
 14: 'parking meter',
 15: 'bench',
 16: 'bird',
 17: 'cat',
 18: 'dog',
 19: 'horse',
 20: 'sheep',
 21: 'cow',
 22: 'elephant',
 23: 'bear',
 24: 'zebra',
 25: 'giraffe',
 27: 'backpack',
 28: 'umbrella',
 31: 'handbag',
 32: 'tie',
 33: 'suitcase',
 34: 'frisbee',
 35: 'skis',
 36: 'snowboard',
 37: 'sports ball',
 38: 'kite',
 39: 'baseball bat',
 40: 'baseball glove',
 41: 'skateboard',
 42: 'surfboard',
 43: 'tennis racket',
 44: 'bottle',
 46: 'wine glass',
 47: 'cup',
 48: 'fork',
 49: 'knife',
 50: 'spoon',
 51: 'bowl',
 52: 'banana',
 53: 'apple',
 54: 'sandwich',
 55: 'orange',
 56: 'broccoli',
 57: 'carrot',
 58: 'hot dog',
 59: 'pizza',
 60: 'donut',
 61: 'cake',
 62: 'chair',
 63: 'couch',
 64: 'potted plant',
 65: 'bed',
 67: 'dining table',
 70: 'toilet',
 72: 'tv',
 73: 'laptop',
 74: 'mo

In [ ]:
obj = results[images[0]]["detector_output"]
obj.keys()

TypeError: list indices must be integers or slices, not str

In [ ]:
n_detections = int(obj["num_detections"].numpy()[0])
n_detections

100

In [ ]:
result = []
n_detections = int(obj["num_detections"].numpy()[0])

for i in range(n_detections):
     r = [
        {"category_id":obj["detection_classes"].numpy()[0][i],
         "score": obj["detection_scores"].numpy()[0][i],
         "bbox": obj["detection_boxes"].numpy()[0][i]}
     ]
     result.append(r)

result[2]

[{'category_id': 16.0,
  'score': 0.3826751,
  'bbox': array([0.1492896 , 0.4810276 , 0.31278604, 0.837199  ], dtype=float32)}]

In [ ]:
with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)

In [ ]:
filename_to_image_id['COCO_val2014_000000391895.jpg']

391895

In [ ]:
filename_to_image_id= {image['file_name']: image["id"] for image in coco_data['images']}
filename_to_image_id

{'COCO_val2014_000000391895.jpg': 391895,
 'COCO_val2014_000000522418.jpg': 522418,
 'COCO_val2014_000000184613.jpg': 184613,
 'COCO_val2014_000000318219.jpg': 318219,
 'COCO_val2014_000000554625.jpg': 554625,
 'COCO_val2014_000000397133.jpg': 397133,
 'COCO_val2014_000000574769.jpg': 574769,
 'COCO_val2014_000000060623.jpg': 60623,
 'COCO_val2014_000000309022.jpg': 309022,
 'COCO_val2014_000000005802.jpg': 5802,
 'COCO_val2014_000000222564.jpg': 222564,
 'COCO_val2014_000000118113.jpg': 118113,
 'COCO_val2014_000000193271.jpg': 193271,
 'COCO_val2014_000000224736.jpg': 224736,
 'COCO_val2014_000000483108.jpg': 483108,
 'COCO_val2014_000000403013.jpg': 403013,
 'COCO_val2014_000000374628.jpg': 374628,
 'COCO_val2014_000000328757.jpg': 328757,
 'COCO_val2014_000000384213.jpg': 384213,
 'COCO_val2014_000000293802.jpg': 293802,
 'COCO_val2014_000000086408.jpg': 86408,
 'COCO_val2014_000000037777.jpg': 37777,
 'COCO_val2014_000000372938.jpg': 372938,
 'COCO_val2014_000000386164.jpg': 38616

In [ ]:
coco_data["images"]

[{'license': 3,
  'file_name': 'COCO_val2014_000000391895.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000391895.jpg',
  'height': 360,
  'width': 640,
  'date_captured': '2013-11-14 11:18:45',
  'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg',
  'id': 391895},
 {'license': 4,
  'file_name': 'COCO_val2014_000000522418.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000522418.jpg',
  'height': 480,
  'width': 640,
  'date_captured': '2013-11-14 11:38:44',
  'flickr_url': 'http://farm1.staticflickr.com/1/127244861_ab0c0381e7_z.jpg',
  'id': 522418},
 {'license': 3,
  'file_name': 'COCO_val2014_000000184613.jpg',
  'coco_url': 'http://images.cocodataset.org/val2014/COCO_val2014_000000184613.jpg',
  'height': 336,
  'width': 500,
  'date_captured': '2013-11-14 12:36:29',
  'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg',
  'id': 184613},
 {'license': 3,
  'file_name': 'COC